In [1]:
#!pip install python-dotenv sqlalchemy psycopg2-binary pandas

In [2]:
import os
import pandas as pd
import numpy as np
import psycopg2
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [3]:
# Cargar las variables de entorno desde .env
load_dotenv()

# Conexión general
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")

# Base de datos origen
dbname = os.getenv("DB_NAME")

# Bodega de datos
dwname = os.getenv("DW_NAME")

print(f"{dbname} {dwname}")


Base_Datos_Proyecto Bodega_Datos_Proyecto


In [4]:
# Engine para base origen
engine_db = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}")

# Engine para la bodega de datos
engine_dw = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dwname}")


In [4]:
query = "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' AND table_type = 'BASE TABLE';"

df = pd.read_sql(query, engine_dw)

df.head()

,table_name


In [5]:
# 1. Leer la tabla ciudad desde la base original
query = "SELECT * FROM ciudad;"
df_ciudad = pd.read_sql(query, engine_db)

# 2. Renombrar columnas en pandas
df_ciudad = df_ciudad.rename(columns={
    'ciudad_id': 'id',
    'nombre': 'nombre_ciudad'
})

# Aplica encoding UTF-8 forzado a todas las columnas tipo string
for col in df_ciudad.select_dtypes(include='object').columns:
    df_ciudad[col] = df_ciudad[col].apply(lambda x: x.encode('utf-8', errors='replace').decode('utf-8') if isinstance(x, str) else x)


# 3. Guardar el dataframe en la bodega de datos (reemplazando la tabla ciudad)
df_ciudad.to_sql('ciudad', con=engine_dw, if_exists='replace', index=False)

# 4. Mostrar resultado para verificar
df_ciudad.head()


,id,nombre_ciudad,departamento_id
0,6,BUGA,1
1,5,BOGOTA,2
2,4,PASTO,4
3,3,POPAYAN,3
4,2,PALMIRA,1


In [6]:
# 1. Leer la tabla 'departamento' desde la base de datos original
df_departamento = pd.read_sql("SELECT * FROM departamento;", con=engine_db)

# 2. Realizar las transformaciones (renombrar columnas si existen)
df_departamento = df_departamento.rename(columns={
    'departamento_id': 'id',
    'nombre': 'nombre_departamento'
})

# 3. Guardar el dataframe transformado en la bodega de datos (reemplazar si ya existe)
df_departamento.to_sql('departamento', con=engine_dw, if_exists='replace', index=False)

# 4. Verificar
df_departamento.head()


,id,nombre_departamento
0,4,NARIÑO
1,3,CAUCA
2,2,CUNDINAMARCA
3,1,VALLE DEL CAUCA


## Dimensión cliente

In [7]:
# 1. Leer las tablas necesarias desde la base de datos original
df_clientes_usuarioaquitoy = pd.read_sql("SELECT * FROM clientes_usuarioaquitoy;", con=engine_db)
df_ciudad = pd.read_sql("SELECT * FROM ciudad;", con=engine_db)
df_departamento = pd.read_sql("SELECT * FROM departamento;", con=engine_db)
df_cliente = pd.read_sql("SELECT * FROM cliente;", con=engine_db)

# 2. Renombrar columnas necesarias para estandarizar
df_ciudad = df_ciudad.rename(columns={
    'ciudad_id': 'id',
    'nombre': 'nombre_ciudad',
    'departamento_id': 'departamento_id'
})

df_departamento = df_departamento.rename(columns={
    'departamento_id': 'id',
    'nombre': 'nombre_departamento'
})

df_cliente = df_cliente.rename(columns={
    'cliente_id': 'id',
    'nombre': 'nombre_cliente'
})

# 3. Realizar joins con pandas para construir la dimensión cliente
df_dim_cliente = df_clientes_usuarioaquitoy \
    .merge(df_ciudad, left_on='ciudad_id', right_on='id', suffixes=('', '_ciudad')) \
    .merge(df_departamento, left_on='departamento_id', right_on='id', suffixes=('', '_depto')) \
    .merge(df_cliente, left_on='cliente_id', right_on='id', suffixes=('', '_cliente'))

# 4. Seleccionar solo las columnas deseadas
df_dim_cliente = df_dim_cliente[[
    'cliente_id', 'nit_cliente', 'nombre_cliente', 'user_id',
    'nombre_ciudad', 'nombre_departamento'
]]

# 5. Guardar la tabla en la bodega de datos como dimensión cliente
df_dim_cliente.to_sql('dim_cliente', con=engine_dw, if_exists='replace', index=False)

# 6. Verificar
df_dim_cliente.head()


,cliente_id,nit_cliente,nombre_cliente,user_id,nombre_ciudad,nombre_departamento
0,11,327282-4,CARROS DEL PACIFICO (CHINA),166,CALI,VALLE DEL CAUCA
1,11,327282-4,CARROS DEL PACIFICO (CHINA),169,CALI,VALLE DEL CAUCA
2,11,327282-4,CARROS DEL PACIFICO (CHINA),168,CALI,VALLE DEL CAUCA
3,6,24390-3,CLINICA DEPORTIVA DEL SUR,176,CALI,VALLE DEL CAUCA
4,6,24390-3,CLINICA DEPORTIVA DEL SUR,171,CALI,VALLE DEL CAUCA


## Dimensión mensajero

In [8]:
# 1. Leer las tablas necesarias desde la base de datos origen
df_mensajero   = pd.read_sql("SELECT * FROM clientes_mensajeroaquitoy;", con=engine_db)
df_auth_user   = pd.read_sql("SELECT id, username FROM auth_user;", con=engine_db)
df_ciudad      = pd.read_sql("SELECT * FROM ciudad;", con=engine_db)
df_departamento = pd.read_sql("SELECT * FROM departamento;", con=engine_db)

# 2. Renombrar columnas para normalizar nombres
df_ciudad = df_ciudad.rename(columns={
    'ciudad_id': 'id',              # si existe esa columna
    'nombre':    'nombre_ciudad',
    'departamento_id': 'departamento_id'
})

df_departamento = df_departamento.rename(columns={
    'departamento_id': 'id',        # si existe esa columna
    'nombre':          'nombre_departamento'
})

# 3. Construir la dimensión mensajero con merges de pandas
df_dim_mensajero = (
    df_mensajero
    .merge(df_auth_user, left_on='user_id', right_on='id', suffixes=('', '_user'))
    .merge(df_ciudad,  how='left', left_on='ciudad_operacion_id', right_on='id', suffixes=('', '_ciudad'))
    .merge(df_departamento, how='left', left_on='departamento_id', right_on='id', suffixes=('', '_depto'))
)

# 4. Seleccionar y renombrar las columnas finales
df_dim_mensajero = df_dim_mensajero[[
    'id',                 # id del mensajero
    'username',           # nombre de usuario
    'activo',
    'fecha_entrada',
    'nombre_ciudad',
    'nombre_departamento'
]].rename(columns={
    'id': 'mensajero_id',
    'username': 'nombre',
    'nombre_ciudad': 'ciudad_operacion',
    'nombre_departamento': 'departamento_operacion'
})

# 5. Cargar la dimensión en la bodega de datos
df_dim_mensajero.to_sql('dim_mensajero', con=engine_dw, if_exists='replace', index=False)

# 6. Verificar visualizando las primeras filas
df_dim_mensajero.head()


,mensajero_id,nombre,activo,fecha_entrada,ciudad_operacion,departamento_operacion
0,1,mensajero1,True,None,ACOPI YUMBO,VALLE DEL CAUCA
1,42,JPEDROZA,True,None,CALI,VALLE DEL CAUCA
2,48,JULIANVILLANUEVA,True,2024-07-12,CALI,VALLE DEL CAUCA
3,41,LUISCARDONA,True,None,CALI,VALLE DEL CAUCA
4,13,GEOVANNY Hidalgo,True,2021-11-08,PASTO,NARIÑO


## Dimensión sede

In [5]:
# 1. Leer las tablas necesarias desde la base de datos origen
df_sede        = pd.read_sql("SELECT * FROM sede;", con=engine_db)
df_ciudad      = pd.read_sql("SELECT * FROM ciudad;", con=engine_db)
df_departamento = pd.read_sql("SELECT * FROM departamento;", con=engine_db)

# 2. Renombrar columnas para mantener consistencia
df_ciudad = df_ciudad.rename(columns={
    'ciudad_id': 'id',                # si aplica
    'nombre': 'nombre_ciudad',
    'departamento_id': 'departamento_id'
})

df_departamento = df_departamento.rename(columns={
    'departamento_id': 'id',          # si aplica
    'nombre': 'nombre_departamento'
})

df_sede = df_sede.rename(columns={
    'nombre': 'nombre_sede'
})

# 3. Construir la dimensión sede haciendo los joins necesarios
df_dim_sede = (
    df_sede
    .merge(df_ciudad, left_on='ciudad_id', right_on='id', suffixes=('', '_ciudad'))
    .merge(df_departamento, left_on='departamento_id', right_on='id', suffixes=('', '_depto'))
)

# 4. Seleccionar y renombrar las columnas finales
df_dim_sede = df_dim_sede[[
    'id',  # id de la sede
    'nombre_sede',
    'direccion',
    'nombre_ciudad',
    'nombre_departamento',
    'cliente_id'
]].rename(columns={
    'id': 'sede_id',
    'nombre_ciudad': 'ciudad_sede',
    'nombre_departamento': 'departamento_sede'
})

# 5. Cargar en la bodega de datos
df_dim_sede.to_sql('dim_sede', con=engine_dw, if_exists='replace', index=False)

# 6. Verificar visualizando las primeras filas
df_dim_sede.head()


,sede_id,nombre_sede,direccion,ciudad_sede,departamento_sede,cliente_id
0,1,FARALLONES /123,Los angeles distrito Latino,CALI,VALLE DEL CAUCA,4
1,1,REMEDIOZ/ 123,Los angeles distrito Latino,CALI,VALLE DEL CAUCA,4
2,1,DIME / LOS ROJOS,Los angeles distrito Latino,CALI,VALLE DEL CAUCA,4
3,1,DESPACHOS / LOS ROJOS,Los angeles distrito Latino,CALI,VALLE DEL CAUCA,4
4,3,POPAYAN BODEGA 28 / A,Los angeles distrito Latino,POPAYAN,CAUCA,11


## Dimensión fecha

In [10]:
# 1. Leer la tabla original desde la base de datos fuente
df_fecha = pd.read_sql("SELECT id, fecha FROM mensajeria_estadosservicio;", con=engine_db)

# 2. Convertir a datetime
df_fecha['fecha'] = pd.to_datetime(df_fecha['fecha'], errors='coerce')  # 'coerce' convierte errores a NaT

# 3. Crear las columnas 'dia' y 'mes' a partir de 'fecha'
df_fecha['dia'] = df_fecha['fecha'].dt.day
df_fecha['mes'] = df_fecha['fecha'].dt.month_name(locale='es_ES')  # o 'es' si no funciona

# 4. Renombrar la tabla como dim_fecha
df_dim_fecha = df_fecha.rename(columns={'id': 'fecha_id'})

# 5. Guardar la dimensión en la bodega de datos
df_dim_fecha.to_sql('dim_fecha', con=engine_dw, if_exists='replace', index=False)

# 6. Mostrar resultado
df_dim_fecha.head()



,fecha_id,fecha,dia,mes
0,1014,2024-01-29,29,Enero
1,1484,2024-01-30,30,Enero
2,2829,2024-02-06,6,Febrero
3,1888,2024-02-01,1,Febrero
4,32312,2024-04-06,6,Abril


## Dimensión hora

In [11]:
# 1. Leer la tabla original desde la base de datos fuente
df_hora = pd.read_sql("SELECT id, hora FROM mensajeria_estadosservicio;", con=engine_db)

# 2. Convertir la columna 'hora' a tipo datetime (solo hora)
df_hora['hora'] = pd.to_datetime(df_hora['hora'], format='%H:%M:%S', errors='coerce').dt.time

# 3. Extraer componentes hora, minuto y segundo
df_hora['hora_completa'] = pd.to_datetime(df_hora['hora'].astype(str))  # para extraer partes

df_hora['hora'] = df_hora['hora_completa'].dt.hour
df_hora['minuto'] = df_hora['hora_completa'].dt.minute
df_hora['segundo'] = df_hora['hora_completa'].dt.second

# 4. Renombrar id si quieres
df_hora = df_hora.rename(columns={'id': 'hora_id'})

# 5. Guardar la dimensión en la bodega de datos
df_hora.to_sql('dim_hora', con=engine_dw, if_exists='replace', index=False)

# 6. Mostrar resultado para verificar
df_hora.head()


C:\Users\cabre\AppData\Local\Temp\ipykernel_15272\2928532796.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_hora['hora_completa'] = pd.to_datetime(df_hora['hora'].astype(str))  # para extraer partes


,hora_id,hora,hora_completa,minuto,segundo
0,1014,1.0,2025-05-25 01:13:32,13.0,32.0
1,1484,18.0,2025-05-25 18:45:12,45.0,12.0
2,2829,11.0,2025-05-25 11:34:04,34.0,4.0
3,1888,14.0,2025-05-25 14:50:39,50.0,39.0
4,32312,16.0,2025-05-25 16:11:21,11.0,21.0
